<a href="https://colab.research.google.com/github/junkyuhufs/Class2026spring/blob/main/NLPforET_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multimodal Data for English Teachers 🎧🖼️
## Sound + Image with Python in Colab

오늘은 텍스트(text)뿐 아니라  
**소리(sound)** 와 **이미지(image)** 도 파이썬으로 다룰 수 있다는 것을 체험해 봅니다.

---

## 오늘 수업 목표 🎯

오늘은 아래 4가지를 이해하면 충분합니다.

1. 텍스트를 소리로 바꾸는 TTS를 실행할 수 있다.  
2. YouTube 영상의 자막/대본을 가져와 볼 수 있다.  
3. 이미지를 불러오고 간단히 바꿀 수 있다.  
4. 이런 기능이 영어수업 자료 제작과 어떻게 연결되는지 이해한다.

---

## 오늘의 큰 흐름 🛠️

### Part 1. Sound data
- gTTS로 텍스트를 음성으로 만들기
- YouTube 자막/대본 가져오기
- 대본 일부를 다시 TTS로 바꿔 보기

### Part 2. Image data
- 이미지 불러오기
- 크기 바꾸기
- 흑백(grayscale)으로 바꾸기
- 이미지 위에 단어/문장 올리기

---

## 영어교사에게 왜 유용할까? 🎓

- **TTS** → 듣기 자료, 발음 모델, dictation 자료
- **YouTube transcript** → 듣기 대본, 핵심 표현 추출
- **Image editing** → 어휘 flashcard, 그림 설명 활동, reading warm-up 자료

In [ ]:
# 필요한 라이브러리를 설치합니다.
# gtts: 텍스트를 음성(mp3)으로 만들기
# pillow: 이미지 처리
# youtube-transcript-api: 유튜브 자막/대본 가져오기

!pip -q install gTTS pillow youtube-transcript-api requests

In [ ]:
# 오늘 사용할 라이브러리들을 불러옵니다.

from gtts import gTTS
from IPython.display import Audio, display
from youtube_transcript_api import YouTubeTranscriptApi
from PIL import Image, ImageOps, ImageDraw, ImageFont
import requests
from io import BytesIO

# Part 1. Sound data 🎧

먼저 텍스트를 소리로 바꿔 보겠습니다.

이것은 **TTS (Text-to-Speech)** 라고 합니다.  
즉, 글자를 음성으로 바꾸는 기술입니다.

In [ ]:
# 아주 간단한 영어 문장을 음성으로 바꿉니다.
# gTTS는 텍스트를 mp3 파일로 저장할 수 있습니다.

text = "Hello students. Today we will learn English with Python."

tts = gTTS(text=text, lang="en")
tts.save("sample_tts.mp3")

print("mp3 file saved!")

### 결과가 의미하는 것 💡

이 코드는 영어 문장을 mp3 파일로 만든 것입니다.

즉,
- **입력(input)** = 영어 문장
- **처리(process)** = TTS 함수
- **출력(output)** = 음성 파일

이런 식으로 소리 데이터도 만들 수 있습니다.

In [ ]:
# 방금 만든 mp3 파일을 코랩에서 재생합니다.

display(Audio("sample_tts.mp3"))

### 결과가 의미하는 것 💡

이제 학생들은 텍스트가 실제 음성으로 바뀌는 것을 바로 들을 수 있습니다.

영어수업에서는 이런 방식으로
- 짧은 듣기 자료
- 발음 모델
- 따라 읽기 자료
를 만들 수 있습니다.

## 1-1. 말하는 속도/문장 내용을 바꾸면? 🔁

여기서는 문장만 바꿔도 새로운 듣기 자료를 바로 만들 수 있습니다.

In [ ]:
# 문장을 바꿔서 새로운 TTS 파일을 만듭니다.

text2 = "Please listen carefully and choose the correct answer."

tts2 = gTTS(text=text2, lang="en")
tts2.save("instruction_tts.mp3")

display(Audio("instruction_tts.mp3"))

### 영어수업 연결 🎓
- listening instruction 만들기
- speaking model sentence 만들기
- dictation 자료 만들기

In [ ]:
## 1-2. YouTube 영상의 자막/대본 가져오기 📺

이번에는 YouTube 영상의 **자막/대본(transcript)** 을 가져와 보겠습니다.

중요한 점:
- 오늘은 **영상 자체를 다운로드**하는 것이 아니라
- **자막/대본 텍스트**를 가져오는 것입니다.

In [ ]:
# 분석할 YouTube 영상 ID를 넣습니다.
# 예시 ID는 바꿔도 됩니다.
# URL이 아니라 "video id"만 넣는 것이 편합니다.

video_id = "dQw4w9WgXcQ"

In [ ]:
# YouTube 자막/대본을 가져옵니다.
# 자막이 없거나 막혀 있으면 에러가 날 수 있습니다.
# 그 경우 다른 영상 ID를 사용하면 됩니다.

try:
    transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=["en"])
    print("Transcript loaded!")
    print(transcript[:5])   # 앞부분 5개만 보기
except Exception as e:
    print("Transcript could not be loaded.")
    print(e)

### 결과가 의미하는 것 💡

결과는 보통 이런 형태입니다.

- `text` : 자막 문장
- `start` : 시작 시간
- `duration` : 지속 시간

즉, YouTube 자막도 **분석 가능한 텍스트 데이터**입니다.

In [ ]:
# transcript를 하나의 긴 문자열로 합칩니다.

try:
    transcript_text = " ".join([item["text"] for item in transcript])
    print(transcript_text[:500])   # 앞부분 500글자만 보기
except:
    transcript_text = ""
    print("No transcript text available.")

### 결과가 의미하는 것 💡

이제 유튜브 자막을 하나의 텍스트처럼 볼 수 있습니다.

즉, 이 대본을 가지고
- 핵심 표현 찾기
- 단어 빈도 보기
- TTS로 다시 읽기
같은 작업이 가능합니다.

In [ ]:
# YouTube 대본 일부를 다시 TTS로 바꿔 봅니다.
# 너무 길면 부담스러우니 앞부분 일부만 사용합니다.

sample_transcript = transcript_text[:200]

if sample_transcript.strip():
    tts3 = gTTS(text=sample_transcript, lang="en")
    tts3.save("youtube_sample_tts.mp3")
    display(Audio("youtube_sample_tts.mp3"))
else:
    print("Transcript text is empty.")

### 결과가 의미하는 것 💡

이 단계는
**YouTube transcript → text → TTS audio**
로 다시 바꾼 것입니다.

즉, 하나의 멀티모달 흐름입니다.

- 영상의 자막을 텍스트로 보고
- 그 일부를 다시 음성으로 바꿔서 활용할 수 있습니다.

# Part 2. Image data 🖼️

이번에는 이미지를 다뤄 보겠습니다.

오늘은 아래를 해 봅니다.

- 이미지 불러오기
- 크기 바꾸기
- 흑백으로 바꾸기
- 이미지 위에 텍스트 올리기

In [ ]:
# 웹에서 공개 이미지 하나를 불러옵니다.
# 원하는 다른 이미지 URL로 바꿔도 됩니다.

image_url = "https://upload.wikimedia.org/wikipedia/commons/0/0f/Grosser_Panda.JPG"
response = requests.get(image_url)
img = Image.open(BytesIO(response.content))

img

### 결과가 의미하는 것 💡

이제 웹의 이미지를 파이썬으로 불러왔습니다.

즉,
- **입력** = 이미지 URL
- **처리** = 이미지 열기
- **출력** = 화면에 표시되는 이미지

In [ ]:
# 이미지의 기본 정보를 확인합니다.

print("Image size:", img.size)   # (가로, 세로)
print("Image mode:", img.mode)   # RGB, L 등

### 결과가 의미하는 것 💡

- `size`는 이미지의 가로/세로 크기입니다.
- `mode`는 색상 방식입니다.
  - `RGB` = 컬러 이미지
  - `L` = grayscale(흑백)

즉, 이미지는 숫자로도 설명될 수 있는 데이터입니다.

In [ ]:
# 이미지 크기를 줄입니다.
# resize()는 아주 기본적인 이미지 처리 기능입니다.

small_img = img.resize((200, 200))
small_img

### 결과가 의미하는 것 💡

이미지 크기를 바꾼 것입니다.

영어수업에서는 이런 방식으로
- flashcard용 그림 크기 맞추기
- 슬라이드용 이미지 정리하기
에 활용할 수 있습니다.

In [ ]:
# 이미지 위에 텍스트를 올립니다.
# 영어수업에서는 그림 카드나 어휘 자료 제작에 유용합니다.

img_with_text = img.copy()
draw = ImageDraw.Draw(img_with_text)

# 기본 폰트 사용
draw.text((20, 20), "panda", fill="red")

img_with_text

### 결과가 의미하는 것 💡

이제 이미지 위에 단어를 직접 올렸습니다.

영어교사에게는 이것이 바로
- vocabulary card
- picture prompt
- warm-up material
로 연결될 수 있습니다.

In [ ]:
# 조금 더 수업다운 예시:
# 그림 위에 간단한 질문을 올려봅니다.

img_question = img.copy()
draw = ImageDraw.Draw(img_question)

draw.text((20, 20), "What can you see?", fill="blue")

img_question

### 영어수업 연결 🎓
- picture description activity
- vocabulary flashcards
- speaking prompt cards
- writing warm-up images

# 오늘 수업 정리 ✅

오늘은 sound와 image를 아주 기초적으로 다뤄 보았습니다.

### Sound
- 텍스트를 TTS로 음성으로 바꾸기
- YouTube 자막/대본 가져오기
- 대본 일부를 다시 음성으로 바꾸기

### Image
- 이미지 불러오기
- 크기 바꾸기
- 흑백으로 바꾸기
- 이미지 위에 단어/문장 올리기

---

## 오늘의 핵심 메시지 💡

멀티모달 데이터도 결국
**입력 → 처리 → 출력**
의 흐름으로 이해할 수 있습니다.

- 입력: 텍스트 / 유튜브 자막 / 이미지
- 처리: 함수(function) 적용
- 출력: 음성 / 정리된 텍스트 / 수정된 이미지

# 영어교사에게 특히 유용한 추가 아이디어 🚀

오늘 배운 것 외에 영어교사에게 유용한 sound / image 활용은 아래와 같습니다.

### Sound
- 짧은 listening dictation 파일 만들기
- 문장별 발음 모델 만들기
- 학생용 shadowing 자료 만들기

### Image
- 그림 + 단어 flashcard 만들기
- picture description prompt 자동화
- 이미지에 핵심 어휘/질문 넣기

즉, 멀티모달 데이터를 다루는 기술은  
그 자체가 목표가 아니라 **수업 자료 제작 능력**과 연결됩니다.